In [10]:
# 02 - Bronze: ingesta real de sentimiento (Reddit)
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import json

# Si ejecutas en Colab
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_ROOT = Path('/content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2')
BRONZE_SENTIMENT = PROJECT_ROOT / 'data/bronze/sentiment_data'
BRONZE_SENTIMENT.mkdir(parents=True, exist_ok=True)

SENTIMENT_MAP = {
    'muy_negativo': -2,
    'negativo': -1,
    'neutro': 0,
    'positivo': 1,
    'muy_positivo': 2,
}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BRONZE_SENTIMENT:', BRONZE_SENTIMENT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2
BRONZE_SENTIMENT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/bronze/sentiment_data


In [11]:
# Dependencias para Reddit + sentimiento
!pip -q install praw nltk feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.1 MB/s eta 0:00:00


In [12]:
# Configurar acceso a Reddit (con credenciales o modo público sin credenciales)
# Opción recomendada en Colab Secrets: REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USER_AGENT

import os
import requests
import praw
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

def _get_secret(name: str, default: str = '') -> str:
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name, default)

REDDIT_CLIENT_ID = _get_secret('REDDIT_CLIENT_ID')
REDDIT_CLIENT_SECRET = _get_secret('REDDIT_CLIENT_SECRET')
REDDIT_USER_AGENT = _get_secret('REDDIT_USER_AGENT', 'btc-sentiment-script/0.1')

reddit = None
REDDIT_MODE = 'public_json'

if REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET:
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT,
        check_for_async=False,
    )
    REDDIT_MODE = 'praw_api'
    print('Modo Reddit: API autenticada (PRAW).')
else:
    print('Modo Reddit: público sin credenciales (JSON endpoint).')
    print('Tip: agrega credenciales en Colab Secrets para extraer también comentarios.')

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Modo Reddit: público sin credenciales (JSON endpoint).
Tip: agrega credenciales en Colab Secrets para extraer también comentarios.


In [13]:
# Utilidades de etiquetado y guardado Bronze
def compound_to_label(score: float) -> str:
    if score >= 0.60:
        return 'muy_positivo'
    if score >= 0.15:
        return 'positivo'
    if score <= -0.60:
        return 'muy_negativo'
    if score <= -0.15:
        return 'negativo'
    return 'neutro'

def score_text(text: str):
    s = sia.polarity_scores(str(text))
    label = compound_to_label(s['compound'])
    return s['compound'], label, SENTIMENT_MAP[label]

def save_bronze(df: pd.DataFrame, out_dir: Path, base_name: str, metadata: dict):
    csv_path = out_dir / f'{base_name}.csv'
    json_path = out_dir / f'{base_name}.json'
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f'OK -> {csv_path.name} ({len(df):,} filas)')
    print(f'OK -> {json_path.name}')

In [14]:
# Ingesta de Reddit (PRAW -> JSON público -> RSS fallback)
import feedparser

subreddits = ['Bitcoin', 'CryptoCurrency', 'BitcoinMarkets']
post_limit_per_subreddit = 150
top_comments_per_post = 5

rows = []
utc_now = datetime.now(timezone.utc).isoformat()

# 1) Modo autenticado (PRAW)
if REDDIT_MODE == 'praw_api':
    for sub_name in subreddits:
        sub = reddit.subreddit(sub_name)
        print(f'Extrayendo r/{sub_name} con PRAW ...')
        for post in sub.new(limit=post_limit_per_subreddit):
            created_post = datetime.fromtimestamp(post.created_utc, tz=timezone.utc)
            text_post = f"{post.title} {post.selftext or ''}".strip()
            compound, label, ordinal = score_text(text_post)

            rows.append({
                'source': 'reddit',
                'subreddit': sub_name,
                'content_type': 'post',
                'content_id': post.id,
                'parent_id': None,
                'author': str(post.author) if post.author else None,
                'timestamp': created_post.isoformat(),
                'title': post.title,
                'text': post.selftext,
                'score_social': post.score,
                'num_comments': post.num_comments,
                'url': post.url,
                'sentiment_compound': compound,
                'sentiment_label': label,
                'sentiment_ordinal': ordinal,
            })

            post.comments.replace_more(limit=0)
            for c in post.comments[:top_comments_per_post]:
                created_com = datetime.fromtimestamp(c.created_utc, tz=timezone.utc)
                compound_c, label_c, ordinal_c = score_text(c.body)
                rows.append({
                    'source': 'reddit',
                    'subreddit': sub_name,
                    'content_type': 'comment',
                    'content_id': c.id,
                    'parent_id': post.id,
                    'author': str(c.author) if c.author else None,
                    'timestamp': created_com.isoformat(),
                    'title': post.title,
                    'text': c.body,
                    'score_social': c.score,
                    'num_comments': None,
                    'url': f'https://www.reddit.com{post.permalink}',
                    'sentiment_compound': compound_c,
                    'sentiment_label': label_c,
                    'sentiment_ordinal': ordinal_c,
                })

# 2) Fallback JSON público si no hay filas
if len(rows) == 0:
    headers = {'User-Agent': REDDIT_USER_AGENT}
    per_sub = min(post_limit_per_subreddit, 100)
    for sub_name in subreddits:
        url = f'https://www.reddit.com/r/{sub_name}/new.json?limit={per_sub}'
        print(f'Extrayendo r/{sub_name} en modo JSON público ...')
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            if resp.status_code != 200:
                print(f'No se pudo leer {sub_name}: status={resp.status_code}')
                continue

            payload = resp.json()
            children = payload.get('data', {}).get('children', [])
            for item in children:
                p = item.get('data', {})
                created_utc = p.get('created_utc')
                if created_utc is None:
                    continue
                created_post = datetime.fromtimestamp(created_utc, tz=timezone.utc)
                title = p.get('title', '')
                selftext = p.get('selftext', '') or ''
                text_post = f"{title} {selftext}".strip()
                compound, label, ordinal = score_text(text_post)

                rows.append({
                    'source': 'reddit',
                    'subreddit': sub_name,
                    'content_type': 'post',
                    'content_id': p.get('id'),
                    'parent_id': None,
                    'author': p.get('author'),
                    'timestamp': created_post.isoformat(),
                    'title': title,
                    'text': selftext,
                    'score_social': p.get('score'),
                    'num_comments': p.get('num_comments'),
                    'url': f"https://www.reddit.com{p.get('permalink', '')}",
                    'sentiment_compound': compound,
                    'sentiment_label': label,
                    'sentiment_ordinal': ordinal,
                })
        except Exception as e:
            print(f'Error JSON público en r/{sub_name}: {str(e)[:160]}')

# 3) Fallback RSS si sigue vacío
if len(rows) == 0:
    per_sub = min(post_limit_per_subreddit, 100)
    for sub_name in subreddits:
        rss_url = f'https://www.reddit.com/r/{sub_name}/new/.rss?limit={per_sub}'
        print(f'Extrayendo r/{sub_name} en modo RSS ...')
        feed = feedparser.parse(rss_url)
        for entry in feed.entries:
            published = entry.get('published', None)
            ts = pd.to_datetime(published, utc=True, errors='coerce')
            if pd.isna(ts):
                continue

            title = entry.get('title', '')
            summary = entry.get('summary', '') or ''
            text_post = f"{title} {summary}".strip()
            compound, label, ordinal = score_text(text_post)

            entry_id = entry.get('id', '')
            permalink = entry.get('link', '')

            rows.append({
                'source': 'reddit_rss',
                'subreddit': sub_name,
                'content_type': 'post',
                'content_id': entry_id[-12:] if entry_id else None,
                'parent_id': None,
                'author': entry.get('author', None),
                'timestamp': ts.isoformat(),
                'title': title,
                'text': summary,
                'score_social': None,
                'num_comments': None,
                'url': permalink,
                'sentiment_compound': compound,
                'sentiment_label': label,
                'sentiment_ordinal': ordinal,
            })

reddit_df = pd.DataFrame(rows)
if reddit_df.empty:
    raise ValueError('No se pudo extraer data de Reddit ni por API/JSON/RSS. Intenta más tarde o cambia red.')

reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'], utc=True)
reddit_df = reddit_df.sort_values('timestamp').reset_index(drop=True)

mode_used = 'praw_api' if (REDDIT_MODE == 'praw_api' and len(rows) > 0) else 'public_json_or_rss'

meta_reddit = {
    'dataset': 'reddit_sentiment_raw',
    'source': 'reddit',
    'mode': mode_used,
    'subreddits': subreddits,
    'post_limit_per_subreddit': post_limit_per_subreddit,
    'top_comments_per_post': top_comments_per_post if mode_used == 'praw_api' else 0,
    'downloaded_at_utc': utc_now,
    'rows': int(len(reddit_df)),
    'columns': list(reddit_df.columns),
    'sentiment_map': SENTIMENT_MAP,
}

save_bronze(reddit_df, BRONZE_SENTIMENT, 'bitcoin_news_reddit', meta_reddit)

Extrayendo r/Bitcoin en modo JSON público ...
No se pudo leer Bitcoin: status=403
Extrayendo r/CryptoCurrency en modo JSON público ...
No se pudo leer CryptoCurrency: status=403
Extrayendo r/BitcoinMarkets en modo JSON público ...
No se pudo leer BitcoinMarkets: status=403
Extrayendo r/Bitcoin en modo RSS ...
Extrayendo r/CryptoCurrency en modo RSS ...
Extrayendo r/BitcoinMarkets en modo RSS ...
OK -> bitcoin_news_reddit.csv (300 filas)
OK -> bitcoin_news_reddit.json


In [15]:
# Serie temporal agregada de sentimiento (1h y 1d)
agg_1h = (
    reddit_df
    .set_index('timestamp')
    .groupby([pd.Grouper(freq='1h'), 'subreddit'])
    .agg(
        sentiment_compound_mean=('sentiment_compound', 'mean'),
        sentiment_ordinal_mean=('sentiment_ordinal', 'mean'),
        sentiment_ordinal_sum=('sentiment_ordinal', 'sum'),
        mentions=('content_id', 'count'),
    )
    .reset_index()
    .rename(columns={'timestamp': 'bucket_ts'})
)

agg_1d = (
    reddit_df
    .set_index('timestamp')
    .groupby([pd.Grouper(freq='1d'), 'subreddit'])
    .agg(
        sentiment_compound_mean=('sentiment_compound', 'mean'),
        sentiment_ordinal_mean=('sentiment_ordinal', 'mean'),
        sentiment_ordinal_sum=('sentiment_ordinal', 'sum'),
        mentions=('content_id', 'count'),
    )
    .reset_index()
    .rename(columns={'timestamp': 'bucket_ts'})
)

meta_1h = {
    'dataset': 'reddit_sentiment_agg_1h',
    'source': 'reddit',
    'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
    'rows': int(len(agg_1h)),
    'columns': list(agg_1h.columns),
}
meta_1d = {
    'dataset': 'reddit_sentiment_agg_1d',
    'source': 'reddit',
    'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
    'rows': int(len(agg_1d)),
    'columns': list(agg_1d.columns),
}

save_bronze(agg_1h, BRONZE_SENTIMENT, 'bitcoin_news_reddit_agg_1h', meta_1h)
save_bronze(agg_1d, BRONZE_SENTIMENT, 'bitcoin_news_reddit_agg_1d', meta_1d)

OK -> bitcoin_news_reddit_agg_1h.csv (200 filas)
OK -> bitcoin_news_reddit_agg_1h.json
OK -> bitcoin_news_reddit_agg_1d.csv (97 filas)
OK -> bitcoin_news_reddit_agg_1d.json


In [16]:
# Verificación rápida
sent_files = sorted([p.name for p in BRONZE_SENTIMENT.glob('*')])
print('--- BRONZE SENTIMENT ---')
for f in sent_files:
    print(f)

display(reddit_df.head(3))
display(agg_1h.head(3))

--- BRONZE SENTIMENT ---
bitcoin_news_reddit.csv
bitcoin_news_reddit.json
bitcoin_news_reddit_agg_1d.csv
bitcoin_news_reddit_agg_1d.json
bitcoin_news_reddit_agg_1h.csv
bitcoin_news_reddit_agg_1h.json


,source,subreddit,content_type,content_id,parent_id,author,timestamp,title,text,score_social,num_comments,url,sentiment_compound,sentiment_label,sentiment_ordinal
0,reddit_rss,BitcoinMarkets,post,w/t3_1pm5sre,None,/u/AutoModerator,2025-12-14 05:02:19+00:00,"[Daily Discussion] - Sunday, December 14, 2025","<!-- SC_OFF --><div class=""md""><p><strong>Thre...",None,None,https://www.reddit.com/r/BitcoinMarkets/commen...,0.9512,muy_positivo,2
1,reddit_rss,BitcoinMarkets,post,w/t3_1pmhrzu,None,/u/imissusenet,2025-12-14 16:13:24+00:00,"5th Annual ""Guess the Low"" contest","<!-- SC_OFF --><div class=""md""><p>It's the 5th...",None,None,https://www.reddit.com/r/BitcoinMarkets/commen...,-0.5475,negativo,-1
2,reddit_rss,BitcoinMarkets,post,w/t3_1pmz282,None,/u/AutoModerator,2025-12-15 05:02:09+00:00,"[Daily Discussion] - Monday, December 15, 2025","<!-- SC_OFF --><div class=""md""><p><strong>Thre...",None,None,https://www.reddit.com/r/BitcoinMarkets/commen...,0.9512,muy_positivo,2


,bucket_ts,subreddit,sentiment_compound_mean,sentiment_ordinal_mean,sentiment_ordinal_sum,mentions
0,2025-12-14 05:00:00+00:00,BitcoinMarkets,0.9512,2.0,2,1
1,2025-12-14 16:00:00+00:00,BitcoinMarkets,-0.5475,-1.0,-1,1
2,2025-12-15 05:00:00+00:00,BitcoinMarkets,0.9512,2.0,2,1
